# 🗄️ Vector Stores

## What is a Vector Store?
A **vector store** (vector database) stores embeddings and lets you search them by semantic similarity.

Traditional DB: `WHERE name = 'Alice'` (exact match)
Vector DB: `WHERE embedding SIMILAR TO question_embedding` (semantic match)

## Popular Vector Stores
| Store | Type | Best For |
|-------|------|---------|
| FAISS | In-memory | Development, small datasets |
| Chroma | Local disk | Development, medium datasets |
| Pinecone | Cloud | Production, large scale |
| Weaviate | Cloud/Self-hosted | Production with filtering |
| pgvector | PostgreSQL extension | If you already use Postgres |
| Qdrant | Cloud/Self-hosted | Production, high performance |

## The RAG Pipeline
```
Documents → Embed → Store in Vector DB
                              ↓
User Question → Embed → Similarity Search → Top-K Documents → LLM
```


In [ ]:
# ── FAISS: In-Memory Vector Store ──────────────────────────────────────
# FAISS = Facebook AI Similarity Search
# WHY FAISS first? No setup, runs in memory, great for learning.
# pip install faiss-cpu langchain-community

from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document

# Sample documents
docs = [
    Document(page_content='Python uses indentation for code blocks.', metadata={'topic': 'python'}),
    Document(page_content='JavaScript uses curly braces for code blocks.', metadata={'topic': 'js'}),
    Document(page_content='LangChain simplifies LLM application development.', metadata={'topic': 'langchain'}),
    Document(page_content='Vector stores enable semantic search.', metadata={'topic': 'vectordb'}),
    Document(page_content='Embeddings convert text to numerical vectors.', metadata={'topic': 'embeddings'}),
]

embeddings = OpenAIEmbeddings(model='text-embedding-3-small')

# Create vector store from documents
# This embeds all documents and stores them in FAISS
vectorstore = FAISS.from_documents(docs, embeddings)

print('Vector store created!')
print(f'Documents stored: {vectorstore.index.ntotal}')

In [ ]:
# ── Similarity Search ────────────────────────────────────────────────────
# This is the CORE operation of RAG.
# Embed the query → find k most similar document vectors.

query = 'How does Python handle code structure?'

# similarity_search: returns Documents (without scores)
results = vectorstore.similarity_search(query, k=2)
print(f'Query: {query}')
print(f'\nTop {len(results)} results:')
for i, doc in enumerate(results):
    print(f'  [{i+1}] {doc.page_content}')
    print(f'       Metadata: {doc.metadata}')

# similarity_search_with_score: returns (Document, score) tuples
results_with_scores = vectorstore.similarity_search_with_score(query, k=3)
print(f'\nWith similarity scores (lower L2 = more similar):')
for doc, score in results_with_scores:
    print(f'  score={score:.4f} | {doc.page_content[:60]}')

In [ ]:
# ── Saving and Loading Vector Stores ────────────────────────────────────
# WHY: You don't want to re-embed documents every time you restart.
# Save embeddings to disk → load them quickly next time.

import os

# Save FAISS index
save_path = '/tmp/my_vectorstore'
vectorstore.save_local(save_path)
print(f'Saved to: {save_path}')
print(f'Files: {os.listdir(save_path)}')

# Load it back
embeddings = OpenAIEmbeddings(model='text-embedding-3-small')
loaded_vectorstore = FAISS.load_local(
    save_path,
    embeddings,
    allow_dangerous_deserialization=True  # required for security acknowledgment
)

# Verify it works
results = loaded_vectorstore.similarity_search('semantic search', k=1)
print(f'\nLoaded store works! Found: {results[0].page_content}')

## 🎯 Interview Questions

1. **What is a vector store and how does it differ from a traditional database?**
2. **What is FAISS? When would you use it vs Pinecone?**
3. **What is the difference between similarity_search() and similarity_search_with_score()?**
4. **Why would you save a vector store to disk?**
5. **How does a vector store find similar documents?**

> Answer these before moving to the next module.

## 💪 Mini Exercise

**Exercise**: Based on what you learned in this module:

1. Re-run all code cells with your own inputs
2. Modify one code cell to add new functionality
3. Answer the interview questions above from memory

**Mini Project**: Build a small application that combines the concepts from this module.


## 📚 Summary

In this module you learned:
- The core concepts of **Vector Stores — Storing and Searching Embeddings**
- How to implement them with modern LangChain/LangGraph APIs
- Common mistakes and how to avoid them
- Interview-level understanding of why each component exists

**Next**: Continue to the next module to build on these foundations.

---
*Part of the LangChain & LangGraph Master Course*
